In [1]:
from river import datasets
from river.datasets import synth
from river import evaluate
from river import metrics
from river.drift import ADWIN
from src.streaming_random_patches import SRPClassifier, SRPClassifierSDDM
from src.river_sddm import RiverSDDM

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# 1. Set up a Synthetic Stream with abrupt Concept Drift
# We transition from SEA generator function 0 to function 2 at instance 2000
stream = synth.ConceptDriftStream(
    stream=synth.SEA(seed=42, variant=0),
    drift_stream=synth.SEA(seed=42, variant=2),
    position=2000,
    width=50,
    seed=123
).take(4000) # Run for 4000 instances

In [4]:
def make_sddm():
    return RiverSDDM(
        n_bins=10,
        ref_window_size=400,
        cur_window_size=100,
        threshold=0.6
    )

model = SRPClassifierSDDM(
    n_models=10,
    n_clusters=3,
    seed=42,
    sddm_constructor=make_sddm
)

In [5]:
# 3. Track multiple metrics
metric = metrics.Accuracy() + metrics.MacroF1()

In [6]:
# 4. Manual Evaluation Loop (Better for debugging than progressive_val_score)
drifts_detected = 0

for i, (x, y) in enumerate(stream):
    # Predict
    y_pred = model.predict_one(x)
    
    # Update metric
    if y_pred is not None:
        metric.update(y, y_pred)
        
    # Train
    model.learn_one(x, y)

    # Print progress every 1000 instances
    if (i + 1) % 1000 == 0:
        print(f"Instance {i+1} | {metric}")

Instance 1000 | Accuracy: 94.99%
MacroF1: 93.94%
Instance 2000 | Accuracy: 96.25%
MacroF1: 95.51%
[DRIFT ENSEMBLE] Instance: 2081 | Cluster: 0 | Feature: 0 | Mag: 0.6228
[DRIFT ENSEMBLE] Instance: 2734 | Cluster: 2 | Feature: 1 | Mag: 0.6056
Instance 3000 | Accuracy: 96.43%
MacroF1: 95.56%
[DRIFT ENSEMBLE] Instance: 3609 | Cluster: 1 | Feature: 1 | Mag: 0.6042
Instance 4000 | Accuracy: 96.52%
MacroF1: 95.58%


In [7]:
# 1. Set up a Synthetic Stream with abrupt Concept Drift
# We transition from SEA generator function 0 to function 2 at instance 2000
stream = synth.ConceptDriftStream(
    stream=synth.SEA(seed=42, variant=0),
    drift_stream=synth.SEA(seed=42, variant=2),
    position=2000,
    width=50,
    seed=123
).take(4000) # Run for 4000 instances
def make_sddm():
    return RiverSDDM(
        n_bins=10,
        ref_window_size=400,
        cur_window_size=100,
        threshold=0.6
    )

model = SRPClassifierSDDM(
    n_models=10,
    n_clusters=2,
    seed=42,
    sddm_constructor=make_sddm
)

In [8]:
# 4. Manual Evaluation Loop (Better for debugging than progressive_val_score)
drifts_detected = 0

for i, (x, y) in enumerate(stream):
    # Predict
    y_pred = model.predict_one(x)
    
    # Update metric
    if y_pred is not None:
        metric.update(y, y_pred)
        
    # Train
    model.learn_one(x, y)

    # Print progress every 1000 instances
    if (i + 1) % 1000 == 0:
        print(f"Instance {i+1} | {metric}")

Instance 1000 | Accuracy: 96.16%
MacroF1: 95.16%
Instance 2000 | Accuracy: 96.38%
MacroF1: 95.50%
[DRIFT ENSEMBLE] Instance: 2167 | Cluster: 0 | Feature: 0 | Mag: 0.6017
Instance 3000 | Accuracy: 96.44%
MacroF1: 95.52%
Instance 4000 | Accuracy: 96.47%
MacroF1: 95.51%
